In [ ]:
!git clone https://github.com/cmorregof/labs-calculo-2026-2.git

Cloning into 'labs-calculo-2026-2'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 88 (delta 31), reused 81 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 32.25 MiB | 35.89 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [ ]:
%cd labs-calculo-2026-2/datos
!ls

/content/labs-calculo-2026-2/datos
agregar_londres.py	     congelar_sipsa.py	    momentum_final2022.csv
arecibo.txt		     eventos_final2022.csv  preparar_momentum.py
CHECKSUMS.md5		     london_energy.csv.gz   README.md
chec_tarifas_congeladas.csv  londres_mensual.csv    sipsa_mensual.csv.gz


In [ ]:
!cp sipsa_mensual.csv.gz sipsa_mensual_profe.csv.gz

In [ ]:
!cat README.md

# Datos de los laboratorios

| Archivo | Qué es | Fuente | Congelado |
|---|---|---|---|
| `londres_mensual.csv` | Consumo **mensual** por hogar de Londres (`hogar, mes, kwh`), 113 440 filas, 5 556 hogares, 2011-12 a 2014-02 | Producido con `agregar_londres.py` a partir de `london_energy.csv` | sept. 2026 |
| `london_energy.csv.gz` | Consumo **diario** por hogar (`LCLid, Date, KWH`), 3 510 433 filas, 5 566 hogares | Kaggle, *London Homes Energy Data* (emmanuelfwerr) — proyecto Low Carbon London, UK Power Networks | sept. 2026 |
| `chec_tarifas_congeladas.csv` | Pliego tarifario CHEC, 2008 a agosto de 2026 (40 294 filas) | datos.gov.co, dataset `c698-wqig` — *Costo unitario de tarifas de energía CHEC* | 28 sept. 2026 |
| `agregar_londres.py` | Script que produce el mensual a partir del diario (regla: meses con ≥ 25 días de lectura) | curso | — |
| `arecibo.txt` | Mensaje de Arecibo (16 nov. 1974): una línea de 1 679 caracteres `0`/`1` (397 unos). Lo usa Introducción a las CC, L1 *Primer

In [ ]:
!python congelar_sipsa.py /tmp/sipsa_mensual.csv.gz

Descargando 2013-2017…
Descargando 2018…
Descargando 2019…
Descargando 2020…
Descargando 2021…
Descargando 2022…
Descargando 2023…
Descargando 2024…
Promediando 8 filas repetidas de la fuente oficial.
Escrito: /tmp/sipsa_mensual.csv.gz
Filas: 666865 | Meses: 2013-01 a 2024-12 (144)
Ciudades: 71 | Productos: 385
Filas por año: {'2013': 54676, '2014': 56334, '2015': 55958, '2016': 57026, '2017': 57392, '2018': 57121, '2019': 56626, '2020': 55716, '2021': 54520, '2022': 53647, '2023': 53807, '2024': 54042}


In [ ]:
!sed -n 100,140p congelar_sipsa.py

    i_fecha = posicion("fecha")
    i_producto = posicion("producto")
    i_mercado = posicion("mercado", "fuente")
    i_precio = next(i for nombre, i in posiciones.items() if nombre.startswith("precio"))

    filas = []
    for fila in lector:
        if not fila or len(fila) <= max(i_fecha, i_producto, i_mercado, i_precio):
            continue
        fecha = fecha_normalizada(fila[i_fecha])
        mercado = fila[i_mercado].strip()
        producto = fila[i_producto].strip()
        if not mercado or not producto:
            continue
        filas.append({
            "fecha": fecha,
            "ciudad": ciudad_de_mercado(mercado),
            "mercado": mercado,
            "producto": producto,
            "precio_kg": precio_normalizado(fila[i_precio]),
        })
    return filas


def descargar(identificador: str) -> bytes:
    solicitud = urllib.request.Request(
        URL.format(identificador), headers={"User-Agent": "labs-calculo-2026-2"}
    )
    with urllib.request.u

In [ ]:
!sed -n 1,60p congelar_sipsa.py

#!/usr/bin/env python3
"""Construye el congelado mensual SIPSA-P 2013--2024 del Lab 2.

La fuente oficial publicada en datos.gov.co (conjunto gkqq-k3k5) es un
catálogo federado: sus ocho archivos ZIP anuales viven en Microdatos DANE.
Cada CSV usa nombres de columnas y formatos de fecha ligeramente distintos.
Este script los normaliza a fecha, ciudad, mercado, producto y precio_kg.

Uso:
    python3 congelar_sipsa.py [salida.csv.gz]
"""

from __future__ import annotations

import csv
import gzip
import io
import sys
import unicodedata
import urllib.request
import zipfile
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path


FUENTES = {
    "2013-2017": "23898",
    "2018": "23897",
    "2019": "23896",
    "2020": "23895",
    "2021": "23894",
    "2022": "23893",
    "2023": "23892",
    "2024": "23891",
}
URL = "https://microdatos.dane.gov.co/index.php/catalog/776/download/{}"
MESES = {
    "ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 

In [ ]:
!curl -I -L -A "labs-calculo-2026-2" "https://microdatos.dane.gov.co/index.php/catalog/776/download/23898"
!curl -I -L -A "Mozilla/5.0" "https://microdatos.dane.gov.co/index.php/catalog/776/download/23898"

HTTP/1.1 200 OK
Date: Fri, 06 Nov 2026 03:03:52 GMT
Content-Type: application/octet-stream
Content-Length: 15664363
Connection: keep-alive
X-Powered-By: PHP/7.4.33
Content-Disposition: attachment; filename="BaseDatos-SIPSA_P-Mensual-2013_2017.zip"
Expires: Mon, 26 Jul 1997 05:00:00 GMT
Accept-Ranges: bytes
Cache-control: private
Pragma: private
Set-Cookie: ihsn_nada=mu689ocv6d056o88437bt84rtdd0ssu5; expires=Fri, 06-Nov-2026 05:03:52 GMT; Max-Age=7200; path=/; HttpOnly
Vary: User-Agent

HTTP/1.1 200 OK
Date: Fri, 06 Nov 2026 03:03:52 GMT
Content-Type: application/octet-stream
Content-Length: 15664363
Connection: keep-alive
X-Powered-By: PHP/7.4.33
Content-Disposition: attachment; filename="BaseDatos-SIPSA_P-Mensual-2013_2017.zip"
Expires: Mon, 26 Jul 1997 05:00:00 GMT
Accept-Ranges: bytes
Cache-control: private
Pragma: private
Set-Cookie: ihsn_nada=d795281pdoghshqiuf6g1lqa0v0gt842; expires=Fri, 06-Nov-2026 05:03:52 GMT; Max-Age=7200; path=/; HttpOnly
Vary: User-Agent



In [ ]:
!curl -s -A "Mozilla/5.0" "https://microdatos.dane.gov.co/index.php/catalog/776/download/23898"

In [ ]:
!mkdir -p zips

In [ ]:
%pwd
!ls /content

labs-calculo-2026-2  sample_data


In [ ]:
!mv "/content/BaseDatos-SIPSA_P-Mensual-2013_2017.zip" zips/23898.zip
!mv "/content/BaseDatos-SIPSA_P-Mensual-2018.zip" zips/23897.zip
!mv "/content/BaseDatos-SIPSA_P-Mensual-2019.zip" zips/23896.zip
!mv "/content/BaseDatos-SIPSA_P-Mensual-2020.zip" zips/23895.zip
!mv "/content/BaseDatos-SIPSA_P-Mensual-2021.zip" zips/23894.zip
!mv "/content/BaseDatos-SIPSA_P-Mensual-2022.zip" zips/23893.zip
!mv "/content/BaseDatos-SIPSA_P-Mensual-2023.zip" zips/23892.zip
!mv "/content/BaseDatos-SIPSA_P-Mensual-2024.zip" zips/23891.zip

mv: cannot stat '/content/BaseDatos-SIPSA_P-Mensual-2013_2017.zip': No such file or directory
mv: cannot stat '/content/BaseDatos-SIPSA_P-Mensual-2018.zip': No such file or directory
mv: cannot stat '/content/BaseDatos-SIPSA_P-Mensual-2019.zip': No such file or directory
mv: cannot stat '/content/BaseDatos-SIPSA_P-Mensual-2020.zip': No such file or directory
mv: cannot stat '/content/BaseDatos-SIPSA_P-Mensual-2021.zip': No such file or directory
mv: cannot stat '/content/BaseDatos-SIPSA_P-Mensual-2022.zip': No such file or directory
mv: cannot stat '/content/BaseDatos-SIPSA_P-Mensual-2023.zip': No such file or directory
mv: cannot stat '/content/BaseDatos-SIPSA_P-Mensual-2024.zip': No such file or directory


In [ ]:
!ls -l zips

total 0


In [ ]:
!cp congelar_sipsa.py congelar_sipsa_original.py

In [ ]:
!python congelar_sipsa.py /tmp/sipsa_mensual.csv.gz

Descargando 2013-2017…
Descargando 2018…
Descargando 2019…
Descargando 2020…
Descargando 2021…
Descargando 2022…
Descargando 2023…
Descargando 2024…
Promediando 8 filas repetidas de la fuente oficial.
Escrito: /tmp/sipsa_mensual.csv.gz
Filas: 666865 | Meses: 2013-01 a 2024-12 (144)
Ciudades: 71 | Productos: 385
Filas por año: {'2013': 54676, '2014': 56334, '2015': 55958, '2016': 57026, '2017': 57392, '2018': 57121, '2019': 56626, '2020': 55716, '2021': 54520, '2022': 53647, '2023': 53807, '2024': 54042}


In [ ]:
!md5sum sipsa_mensual.csv.gz /tmp/sipsa_mensual.csv.gz

cdb4e211fa11af8aa5c7ac4238493057  sipsa_mensual.csv.gz
46cae2cfb950485944ccfddbeeaf6eb8  /tmp/sipsa_mensual.csv.gz


In [ ]:
!zcat sipsa_mensual.csv.gz | md5sum
!zcat /tmp/sipsa_mensual.csv.gz | md5sum

14dae0860ce3699b5bda5cad8049c75d  -
14dae0860ce3699b5bda5cad8049c75d  -


In [ ]:
import pandas as pd
a = pd.read_csv("sipsa_mensual.csv.gz")
b = pd.read_csv("/tmp/sipsa_mensual.csv.gz")
print(a.shape, b.shape)
print(a.equals(b))

(666865, 5) (666865, 5)
True


In [ ]:
b = pd.read_csv("/tmp/sipsa_mensual.csv.gz")
b[b["producto"].str.contains("hartón verde", na=False)
  & b["mercado"].str.contains("Manizales", na=False)
  & (b["fecha"] == "2024-12")]

,fecha,ciudad,mercado,producto,precio_kg
664696,2024-12,Manizales,"Manizales, Centro Galerías",Plátano dominico hartón verde,2663.0


In [ ]:
import pandas as pd
b = pd.read_csv("/tmp/sipsa_mensual.csv.gz")
p = b[(b["producto"] == "Plátano dominico hartón verde") & (b["fecha"] == "2024-12")]
print(p["precio_kg"].describe())
print(b.groupby("ciudad")["mercado"].nunique().describe())

count       5.000000
mean     2505.600000
std       259.986153
min      2045.000000
25%      2578.000000
50%      2593.000000
75%      2649.000000
max      2663.000000
Name: precio_kg, dtype: float64
count    71.000000
mean      1.577465
std       1.294626
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       8.000000
Name: mercado, dtype: float64


In [ ]:
print(p[["ciudad", "mercado", "precio_kg"]].sort_values("precio_kg"))
print(b.groupby("ciudad")["mercado"].nunique().sort_values(ascending=False).head())

                           ciudad                     mercado  precio_kg
662527                    Armenia             Armenia, Retiro     2045.0
663935  Cartago (Valle del Cauca)   Cartago (Valle del Cauca)     2578.0
665646                    Pereira              Pereira, La 41     2593.0
665743                    Pereira            Pereira, Mercasa     2649.0
664696                  Manizales  Manizales, Centro Galerías     2663.0
ciudad
Bogotá              8
Cali                6
Barranquilla        5
Ipiales (Nariño)    5
Bucaramanga         4
Name: mercado, dtype: int64
